# 03 · Creación del pipeline de ingeniería de características
### Ingrid Carolina Ortíz Mejicanos
Se arma con **sklearn** el pipeline de preprocesamiento usando las decisiones del notebook 02 (guardadas en `models/feature_config.json`). Los operadores están programados en `predict_future_sales_eda/features.py`, incluido el **Mapper** visto en clase. El pipeline recibe solo `shop_id`, `item_id` y `date_block_num`, y devuelve la matriz lista para el modelo.
Resultado: `models/pipeline_preprocesamiento.pkl`.

In [1]:
import sys, json, time, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import joblib
from sklearn.pipeline import Pipeline

warnings.filterwarnings("ignore")
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "raw").exists())
RAW, MODELS = ROOT / "data" / "raw", ROOT / "models"
sys.path.insert(0, str(ROOT))

from predict_future_sales_eda.traducciones import traducir_categorias, traducir_tiendas
from predict_future_sales_eda.features import limpiar_ventas, ventas_mensuales, construir_grid, MAPA_TIENDAS, COLS_ID
from predict_future_sales_eda.features import ConstructorRezagos, ImputadorCategorico, ImputadorMedianaGrupo, ImputadorConstante, TratamientoOutliers
from predict_future_sales_eda.features import TransformacionLog, Mapper, CodificadorMediaObjetivo, CodificadorOneHot, EliminarVariables, Escalador

with open(MODELS / "feature_config.json", encoding="utf-8") as f:
    config = json.load(f)
config["mapper_mes_indice_estacional"] = {int(k): v for k, v in config["mapper_mes_indice_estacional"].items()}
for k, v in config.items():
    print(f"{k}: {v}")

2026-11-24 23:39:35.842 -06:00 | INFO     | predict_future_sales_eda.config:<module>:11 - PROJ_ROOT path is: C:\Users\icome\Documents\predict-future-sales-eda_parte2


mes_ini_train: 12
mes_fin_train: 26
mes_ini_valid: 27
mes_fin_valid: 33
mes_test: 34
rezagos_par: [1, 2, 3, 5, 12]
rezagos_item: [1, 2, 3]
rezagos_tienda: [1]
rezagos_categoria: [1]
imputacion_categorica_moda: ['categoria_grupo', 'ciudad', 'tienda_tipo']
imputacion_numerica_mediana_categoria: ['precio_ultimo_item']
imputacion_numerica_cero: ['venta_par_lag_1', 'venta_par_lag_2', 'venta_par_lag_3', 'venta_par_lag_5', 'venta_par_lag_12', 'venta_item_lag_1', 'venta_item_lag_2', 'venta_item_lag_3', 'venta_tienda_lag_1', 'venta_categoria_lag_1']
outliers_iqr_log: ['precio_ultimo_item']
outliers_percentil_99_5: ['venta_item_lag_1', 'venta_item_lag_2', 'venta_item_lag_3']
transformacion_log: ['precio_ultimo_item', 'venta_item_lag_1', 'venta_item_lag_2', 'venta_item_lag_3']
mapper_mes_indice_estacional: {1: 103.6, 2: 99.0, 3: 108.9, 4: 85.0, 5: 85.5, 6: 92.4, 7: 86.4, 8: 94.8, 9: 96.3, 10: 97.8, 11: 103.4, 12: 147.0}
target_encoding: ['shop_id', 'item_category_id', 'ciudad']
one_hot: ['categor

In [7]:
print("numexpr activo:", pd.get_option("compute.use_numexpr"))

numexpr activo: False


In [2]:
sales = pd.read_csv(RAW / "sales_train.csv")
items = pd.read_csv(RAW / "items.csv")
categories = traducir_categorias(pd.read_csv(RAW / "item_categories.csv"))
shops = traducir_tiendas(pd.read_csv(RAW / "shops.csv"))

items_cat = items[["item_id", "item_category_id"]].merge(categories[["item_category_id", "categoria_grupo"]], on="item_category_id", how="left")
tiendas_cat = shops[["shop_id", "ciudad", "tienda_tipo"]]

df = limpiar_ventas(sales)
historia = ventas_mensuales(df)
grid = construir_grid(df)

train = grid[grid["date_block_num"].between(config["mes_ini_train"], config["mes_fin_train"])]
valid = grid[grid["date_block_num"].between(config["mes_ini_valid"], config["mes_fin_valid"])]
X_train, y_train = train[COLS_ID], train["item_cnt_month"]
X_valid, y_valid = valid[COLS_ID], valid["item_cnt_month"]
print(f"Entrenamiento (meses {config['mes_ini_train']}-{config['mes_fin_train']}): {len(X_train):,} filas")
print(f"Validación    (meses {config['mes_ini_valid']}-{config['mes_fin_valid']}): {len(X_valid):,} filas")

Entrenamiento (meses 12-26): 4,810,730 filas
Validación    (meses 27-33): 1,614,364 filas


## Definición del pipeline
Cada paso corresponde a una fila de la tabla resumen del notebook 02. El orden importa: primero se construyen las variables, luego se imputan, después se tratan outliers y se transforman, y al final se codifican y se escalan.

In [3]:
def crear_pipeline_preprocesamiento(config, historia, items_cat, tiendas_cat):
    pasos = [
        ("rezagos", ConstructorRezagos(historia, items_cat, tiendas_cat, rezagos_par=config["rezagos_par"], rezagos_item=config["rezagos_item"], rezagos_tienda=config["rezagos_tienda"], rezagos_categoria=config["rezagos_categoria"], mapa_tiendas=MAPA_TIENDAS)),
        ("imputacion_categorica", ImputadorCategorico(variables=config["imputacion_categorica_moda"])),
        ("imputacion_numerica_mediana", ImputadorMedianaGrupo(variables=config["imputacion_numerica_mediana_categoria"], grupo="item_category_id")),
        ("imputacion_numerica_cero", ImputadorConstante(variables=config["imputacion_numerica_cero"], valor=0)),
        ("outliers_iqr", TratamientoOutliers(variables=config["outliers_iqr_log"], metodo="iqr_log")),
        ("outliers_percentil", TratamientoOutliers(variables=config["outliers_percentil_99_5"], metodo="percentil", percentil=99.5)),
        ("transformacion_log", TransformacionLog(variables=config["transformacion_log"])),
        ("mapper_mes", Mapper(variables=["mes"], mappings=config["mapper_mes_indice_estacional"])),
        ("target_encoding", CodificadorMediaObjetivo(variables=config["target_encoding"], suavizado=20)),
        ("one_hot", CodificadorOneHot(variables=config["one_hot"])),
        ("eliminar", EliminarVariables(variables=config["eliminar"])),
        ("escalado", Escalador()),
    ]
    return Pipeline(steps=pasos)

preprocesamiento = crear_pipeline_preprocesamiento(config, historia, items_cat, tiendas_cat)
preprocesamiento

,steps,"[('rezagos', ...), ('imputacion_categorica', ...), ...]"
,transform_input,None
,memory,None
,verbose,False
,historia,shop...s x 5 columns]
,items,item_i...s x 3 columns]
,tiendas,shop_id ...vl Física
,rezagos_par,"[1, 2, ...]"
,rezagos_item,"[1, 2, ...]"
,rezagos_tienda,[1]
,rezagos_categoria,[1]


In [4]:
inicio = time.time()
Xt_train = preprocesamiento.fit_transform(X_train, y_train)
print(f"Ajuste + transformación del entrenamiento: {time.time() - inicio:.0f} s")
inicio = time.time()
Xt_valid = preprocesamiento.transform(X_valid)
print(f"Transformación de la validación:            {time.time() - inicio:.0f} s")
print("Dimensión entrenamiento:", Xt_train.shape, "| validación:", Xt_valid.shape)
print("Valores nulos después del pipeline:", int(Xt_train.isna().sum().sum()), "/", int(Xt_valid.isna().sum().sum()))
print("Variables finales:", list(Xt_train.columns))
Xt_train.describe().T[["mean", "std", "min", "max"]].round(2)

Ajuste + transformación del entrenamiento: 17 s
Transformación de la validación:            4 s
Dimensión entrenamiento: (4810730, 41) | validación: (1614364, 41)
Valores nulos después del pipeline: 0 / 0
Variables finales: ['shop_id', 'date_block_num', 'item_category_id', 'ciudad', 'mes', 'venta_par_lag_1', 'venta_par_lag_2', 'venta_par_lag_3', 'venta_par_lag_5', 'venta_par_lag_12', 'venta_item_lag_1', 'venta_item_lag_2', 'venta_item_lag_3', 'venta_tienda_lag_1', 'venta_categoria_lag_1', 'item_nuevo', 'par_nuevo', 'item_meses_desde_primera_venta', 'par_meses_desde_primera_venta', 'precio_ultimo_item', 'categoria_grupo_Accesorios', 'categoria_grupo_Baterías', 'categoria_grupo_Cine', 'categoria_grupo_Consolas', 'categoria_grupo_Entradas', 'categoria_grupo_Envíos', 'categoria_grupo_Juegos Android', 'categoria_grupo_Juegos MAC', 'categoria_grupo_Juegos PC', 'categoria_grupo_Juegos consola', 'categoria_grupo_Libros', 'categoria_grupo_Medios vírgenes', 'categoria_grupo_Música', 'categoria_g

,mean,std,min,max
shop_id,0.0,1.00,-1.10,4.01
date_block_num,0.0,1.00,-1.58,1.68
item_category_id,0.0,0.99,-0.70,38.01
ciudad,-0.0,1.00,-1.28,3.24
mes,0.0,1.00,-1.08,3.17
venta_par_lag_1,0.0,0.98,-0.24,15.61
venta_par_lag_2,0.0,0.98,-0.24,15.42
venta_par_lag_3,0.0,0.98,-0.23,15.42
venta_par_lag_5,-0.0,0.98,-0.23,15.96
venta_par_lag_12,0.0,0.99,-0.20,16.58


In [5]:
prueba = X_valid.sample(1000, random_state=1).reset_index(drop=True)
paso_rezagos = preprocesamiento.named_steps["rezagos"].transform(prueba)
paso_rezagos.loc[:99, "ciudad"] = np.nan
paso_rezagos.loc[100:199, "precio_ultimo_item"] = np.nan
resto = Pipeline(preprocesamiento.steps[1:])
salida = resto.transform(paso_rezagos)
print("Nulos insertados:", int(paso_rezagos[["ciudad", "precio_ultimo_item"]].isna().sum().sum()), "| nulos a la salida:", int(salida.isna().sum().sum()))
print("Moda aprendida para ciudad:", preprocesamiento.named_steps["imputacion_categorica"].moda_["ciudad"])
print("Límites de outliers del precio:", preprocesamiento.named_steps["outliers_iqr"].limites_)

Nulos insertados: 252 | nulos a la salida: 0
Moda aprendida para ciudad: Moscú
Límites de outliers del precio: {'precio_ultimo_item': (24.695312059847797, 5676.874139744113)}


In [6]:
# 1) Control de calidad: ninguna variable debe quedar constante en la validación
constantes = Xt_valid.columns[Xt_valid.std() == 0].tolist()
print("Variables constantes en validación:", constantes if constantes else "ninguna ✔")

# 2) Guardar el pipeline (comprimido)
ruta = MODELS / "pipeline_preprocesamiento.pkl"
joblib.dump(preprocesamiento, ruta, compress=3)
print(f"Guardado: {ruta.name}  ({ruta.stat().st_size / 1e6:,.1f} MB)")

# 3) Cargarlo y comparar con el original sobre los mismos datos
cargado = joblib.load(ruta)
muestra_v = X_valid.head(5000)
a = cargado.transform(muestra_v).to_numpy()
b = preprocesamiento.transform(muestra_v).to_numpy()
c = Xt_valid.head(5000).to_numpy()
print("Cargado vs original:            diferencia máx =", float(np.abs(a - b).max()))
print("Cargado vs Xt_valid (celda 6):  diferencia máx =", float(np.abs(a - c).max()))

Variables constantes en validación: ninguna ✔
Guardado: pipeline_preprocesamiento.pkl  (18.3 MB)
Cargado vs original:            diferencia máx = 0.0
Cargado vs Xt_valid (celda 6):  diferencia máx = 0.0


ninguna variable queda constante en la validación. El pipeline se guardó comprimido en `models/pipeline_preprocesamiento.pkl` (**18.3 MB**, **primer entregable**) y al cargarlo produce exactamente el mismo resultado que el original (diferencia 0.0). Para garantizar resultados idénticos sin importar el tamaño de la tabla, los transformadores calculan directamente con numpy y se desactivó el motor `numexpr` de pandas, que daba resultados inconsistentes con más de un millón de filas.